# Kazakh Fact-Checking Benchmark — прогон в Google Colab

Онлайн-прогон без терминала. Colab имеет открытый интернет, поэтому Groq (Llama) здесь работает.

## Порядок действий
1. Слева нажми значок 🔑 (**Secrets**) и добавь ключи:
   - `LLAMA_API_KEY` — ключ Groq
   - `GEMINI_API_KEY` — ключ Google (по желанию)
   Напротив каждого включи тумблер **Notebook access**.
2. Выполняй ячейки сверху вниз (Shift+Enter).

⚠️ Ключи храни ТОЛЬКО в Secrets. Никогда не вставляй их прямо в ячейки и не коммить в репозиторий.

### 1. Клонируем репозиторий и ставим зависимости
Если репозиторий приватный — раскомментируй строку с токеном (Personal Access Token из GitHub).

In [ ]:
BRANCH = 'claude/kazakh-factcheck-benchmark-e76kd5'
REPO = 'github.com/Tim2190/kazakh-factcheck-benchmark.git'

# Публичный репозиторий:
!git clone --branch $BRANCH https://$REPO

# Приватный репозиторий (вместо строки выше): положи GH_TOKEN в Secrets и раскомментируй:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone --branch $BRANCH https://$tok@$REPO

%cd kazakh-factcheck-benchmark
!pip install -q -r requirements.txt
print('OK')

### 2. Загружаем ключи из Colab Secrets в окружение

In [ ]:
import os
from google.colab import userdata

for name in ['LLAMA_API_KEY', 'GEMINI_API_KEY']:
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'{name}: загружен ({val[:4]}...)')
        else:
            print(f'{name}: пусто — пропускаем')
    except Exception as e:
        print(f'{name}: не найден в Secrets — пропускаем')

### 3. Прогон Llama (через Groq)

In [ ]:
!python scripts/export_dataset.py
!python scripts/run_factcheck.py --model llama
!python scripts/score.py results/llama_leg_text01_run.json

### 4. Прогон Gemini (если у ключа есть квота)
Если увидишь `429 / limit: 0` — у ключа нет бесплатной квоты; прогони Gemini через веб-чат.

In [ ]:
!python scripts/run_factcheck.py --model gemini
!python scripts/score.py results/gemini_leg_text01_run.json

### 5. Скачать результаты
Файлы в `results/` (сырые ответы + вердикты). Скачай и пришли мне — я соберу итоговую таблицу и метрики (Wilson CI, McNemar).

In [ ]:
from google.colab import files
import glob
for f in glob.glob('results/*_run.json'):
    print('скачиваю', f)
    files.download(f)